# Encode 177.321 keyframe bằng ViT-L/14 (issue #39)

**Vì sao làm việc này:** BTC cấp sẵn vector CLIP **ViT-B/32** — model yếu nhất trong họ CLIP. Đo thật trên 17 truy vấn KIS (kho L21, 7.720 keyframe):

| Model | Final Score | R@1 | R@5 |
|---|---|---|---|
| ViT-B/32 (BTC cấp) | 0.612 | 0.294 | 0.588 |
| **ViT-L/14** | **0.694** | 0.412 | **0.706** |
| Fusion cả hai | 0.682 | **0.471** | 0.647 |

Hơn **13%**. Đây là đòn bẩy lớn nhất còn lại — và không tốn một đồng API nào vì đều là model mở.

**Thời gian:** ~30–60 phút trên T4 (Colab miễn phí). Notebook **lưu từng phần**, Colab ngắt giữa chừng thì chạy lại là tiếp tục chỗ cũ, không mất công.

**Trước khi chạy:** Runtime → Change runtime type → **T4 GPU**.

In [ ]:
# === BƯỚC 1: kiểm GPU và cài thư viện ===
import torch
assert torch.cuda.is_available(), (
    'CHƯA BẬT GPU! Runtime -> Change runtime type -> T4 GPU. '
    'Chạy trên CPU sẽ mất ~23 tiếng thay vì ~40 phút.')
print('GPU:', torch.cuda.get_device_name(0))
%pip -q install open_clip_torch faiss-cpu

In [ ]:
# === BƯỚC 2: đường dẫn ===
from google.colab import drive
drive.mount('/content/drive')

import os
# Nơi CHỨA ảnh keyframe. Ảnh phải nằm dạng <video_id>/<n:03d>.jpg
# Nếu keyframe đang ở dạng .zip trên Drive thì chạy BƯỚC 3 để giải nén.
KF_DIR  = '/content/keyframes'
ZIP_DIR = '/content/drive/MyDrive/AIC2026/keyframes_zip'  # chỉnh nếu khác
OUT_DIR = '/content/drive/MyDrive/AIC2026/outputs/l14'
os.makedirs(OUT_DIR, exist_ok=True)
print('OUT_DIR =', OUT_DIR)

In [ ]:
# === BƯỚC 3: giải nén keyframe vào ĐĨA CỤC BỘ của Colab ===
# QUAN TRỌNG: đừng đọc thẳng 177 nghìn file nhỏ từ Drive — Drive chậm hơn
# đĩa cục bộ khoảng 50 lần, riêng khâu đọc ảnh sẽ lâu hơn cả khâu encode.
import glob, os, subprocess, time

os.makedirs(KF_DIR, exist_ok=True)
zips = sorted(glob.glob(f'{ZIP_DIR}/*.zip'))
print(f'{len(zips)} file zip')
t0 = time.time()
for z in zips:
    subprocess.run(['unzip', '-q', '-o', z, '-d', KF_DIR])
    print(f'  {os.path.basename(z)}  ({time.time()-t0:.0f}s)')

# Gỡ thư mục lồng nếu có (keyframes/keyframes/...)
while True:
    muc = [p for p in glob.glob(f'{KF_DIR}/*') if os.path.isdir(p)]
    if len(muc) == 1 and not glob.glob(f'{muc[0]}/*/*.jpg') == []:
        break
    if len(muc) == 1:
        KF_DIR = muc[0]
        continue
    break
print('KF_DIR =', KF_DIR)

In [ ]:
# === BƯỚC 4: liệt kê ảnh và kiểm đủ chưa ===
import glob, os

anh = sorted(glob.glob(f'{KF_DIR}/**/*.jpg', recursive=True))
video = sorted({os.path.basename(os.path.dirname(p)) for p in anh})
print(f'{len(anh):,} ảnh   {len(video)} video')
print('Phải là 177.321 ảnh / 873 video. Thiếu thì tải nốt zip còn lại rồi chạy lại BƯỚC 3.')

# keyframe_id = "<video>#<n:06d>" — n LẤY TỪ TÊN FILE, đếm từ 1, khớp cột n
# của map-keyframes. Đây là quy ước chung của cả hệ thống, đừng đổi.
def kf_id(p):
    return f"{os.path.basename(os.path.dirname(p))}#{int(os.path.splitext(os.path.basename(p))[0]):06d}"

ids = [kf_id(p) for p in anh]
print('ID đầu:', ids[0], ' ID cuối:', ids[-1])

In [ ]:
# === BƯỚC 5: ENCODE, lưu từng phần (chạy lại được nếu Colab ngắt) ===
import json, time
import numpy as np, open_clip, torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset

TEN, PRE = 'ViT-L-14', 'laion2b_s32b_b82k'
SHARD  = 20000   # mỗi phần 20k ảnh, lưu xong mới sang phần sau
BATCH  = 256

model, _, preprocess = open_clip.create_model_and_transforms(TEN, pretrained=PRE)
model = model.cuda().eval().half()   # half precision: nhanh gấp đôi, sai số không đáng kể

class Anh(Dataset):
    def __init__(self, ds): self.ds = ds
    def __len__(self): return len(self.ds)
    def __getitem__(self, i):
        return preprocess(Image.open(self.ds[i]).convert('RGB'))

t0 = time.time()
for s in range(0, len(anh), SHARD):
    f = f'{OUT_DIR}/phan_{s:07d}.npy'
    if os.path.exists(f):
        print(f'bỏ qua {os.path.basename(f)} (đã có)')
        continue
    lo = anh[s:s + SHARD]
    dl = DataLoader(Anh(lo), batch_size=BATCH, num_workers=4, pin_memory=True)
    ra = []
    with torch.no_grad():
        for b in dl:
            ra.append(model.encode_image(b.cuda().half()).float().cpu().numpy())
    np.save(f, np.concatenate(ra).astype('float32'))
    xong = min(s + SHARD, len(anh))
    print(f'{xong:,}/{len(anh):,}  {time.time()-t0:.0f}s  '
          f'(còn ~{(len(anh)-xong)/max(1,xong)*(time.time()-t0)/60:.0f} phút)')
print('XONG encode')

In [ ]:
# === BƯỚC 6: gộp các phần, dựng index, lưu về Drive ===
import faiss, glob, json
import numpy as np

phan = sorted(glob.glob(f'{OUT_DIR}/phan_*.npy'))
X = np.concatenate([np.load(p) for p in phan]).astype('float32')
assert len(X) == len(ids), f'LỆCH: {len(X)} vector nhưng {len(ids)} id'

X = np.ascontiguousarray(X)
faiss.normalize_L2(X)              # BẮT BUỘC: inner product = cosine
index = faiss.IndexFlatIP(X.shape[1])
index.add(X)
faiss.write_index(index, f'{OUT_DIR}/l14.faiss')
json.dump(ids, open(f'{OUT_DIR}/l14_ids.json', 'w'))

print(f'{index.ntotal:,} vector × {X.shape[1]} chiều')
print(f'-> {OUT_DIR}/l14.faiss  '
      f'({os.path.getsize(f"{OUT_DIR}/l14.faiss")/1e6:.0f} MB)')
print('ID đầu:', ids[0], '<- phải kết thúc bằng #000001')
print('\nTải 2 file l14.faiss và l14_ids.json về máy, đặt vào D:/AIC2026/full/index_l14/')

## Gặp lỗi

| Hiện tượng | Cách xử lý |
|---|---|
| `CHƯA BẬT GPU` | Runtime → Change runtime type → T4 GPU |
| Colab ngắt giữa chừng | Chạy lại **BƯỚC 5** — nó bỏ qua các phần đã lưu, tiếp tục chỗ dở |
| `CUDA out of memory` | Giảm `BATCH` xuống 128 hoặc 64 |
| Số ảnh < 177.321 | Thiếu zip. Tải nốt rồi chạy lại BƯỚC 3 và 4 |
| `LỆCH: ... vector nhưng ... id` | Có phần lưu dở. Xoá file `phan_*.npy` cuối cùng rồi chạy lại BƯỚC 5 |

**Đừng đổi quy ước ID.** `<video>#<n:06d>` với `n` lấy từ tên file ảnh (đếm từ 1) là quy ước chung của cả hệ thống — đổi là mọi thứ khác lệch theo mà không có lỗi nào báo ra.